# 05 · Qwen3.5-0.8B 오프라인 액션 증류
02에서 검증·선정한 T1의 고정 action cache를 사용합니다. Teacher는 GPU에 로드하지 않습니다.
손실 = task_weight × 정답 액션 L1 + action_kd_weight × Teacher 액션 L1이며 padding은 제외합니다.
기본 가중치는 각각 1.0인 시작값으로, 튜닝·성능 검증 결과가 아닙니다. hidden feature 증류는 포함하지 않습니다.
입력/초기 모델/seed/sampler/학습 예산은 03 Task-only와 동일하고 결과 폴더는 분리합니다.
**준비 전용:** 설치·다운로드·1-step·overfit·본 학습 플래그는 모두 False입니다.
순서: 실행 스위치/캐시 경로 확인 → 설치 → 기존 Qwen 가중치 확인(없으면 다운로드 스위치)
→ 캐시/데이터 사전 검사 및 1-step·재로딩 → overfit → 결과 확인 후 본 학습.
본 학습은 T1-full-action 또는 T1-full-hidden과 원본 prepared 이미지가 필요합니다.
부분 pilot cache는 episode_limit을 지정한 점검에만 사용할 수 있습니다.
04의 LoRA checkpoint는 현재 02의 cache 생성 도구와 직접 호환되지 않습니다.
실제 GPU 학습과 rollout 성공률은 아직 검증하지 않았습니다.

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/colab/VLADE')  # 세 노트북에서 동일한 저장 폴더를 지정하십시오.
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = ROOT  # src/scripts/configs/experiments/third_party가 바로 들어 있는 실제 프로젝트 루트
EXPECTED_SOURCE_MANIFEST_SHA256 = 'bef4ecfba258a81465a103ae88096ea799bb25954063f9dd0d82339758c96f7f'
def stage_source(project_root: Path, scratch_root: Path, expected_manifest_sha256: str) -> Path:
    """Verify Drive source files and stage them without copying data or weights."""
    project_root = project_root.resolve()
    manifest_relative = Path("experiments/manifests/colab_source.json")
    manifest_path = project_root / manifest_relative
    if not manifest_path.is_file():
        raise FileNotFoundError("Drive의 VLADE 폴더에 src/scripts/configs/experiments/third_party 일반 파일 구조를 먼저 복사하십시오: " + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_sha256:
        raise ValueError("노트북과 colab_source.json 버전이 다릅니다. 같은 버전의 일반 프로젝트 파일을 사용하십시오.")
    manifest = json.loads(manifest_bytes)
    code = scratch_root / ("vlade_colab_" + expected_manifest_sha256[:12])
    code = code.resolve()
    if code.is_relative_to(project_root) or project_root.is_relative_to(code):
        raise ValueError("실행 소스 임시 폴더와 Drive 프로젝트 폴더는 서로 겹칠 수 없습니다.")
    for relative, expected in manifest["files_sha256"].items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
            raise ValueError("잘못된 소스 상대 경로: " + relative)
        source = project_root / path
        target = code / path
        if not source.resolve().is_relative_to(project_root) or not target.resolve().is_relative_to(code):
            raise ValueError("프로젝트 밖의 소스 경로: " + relative)
        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise ValueError("Drive 소스가 없거나 변경되었습니다: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            shutil.copyfile(source, target)
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("복사된 소스 무결성 불일치: " + relative)
    # Publish the manifest only after every listed source file has been verified.
    staged_manifest = code / manifest_relative
    staged_manifest.parent.mkdir(parents=True, exist_ok=True)
    temporary_manifest = staged_manifest.with_suffix(".json.tmp")
    temporary_manifest.write_bytes(manifest_bytes)
    temporary_manifest.replace(staged_manifest)
    return code

CODE = stage_source(PROJECT_ROOT, Path('/content'), EXPECTED_SOURCE_MANIFEST_SHA256)
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('Drive:', ROOT, '\n실행 소스:', CODE)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'), HF_TOKEN_PATH='/content/vlade_hf_token')
def run(args, env=None):
    import signal, threading, time
    deadline = float((env or ENV).get('VLADE_DEADLINE_UNIX', '0'))
    if deadline and deadline - time.time() <= 60:
        raise RuntimeError('8시간 예산이 끝났습니다. 새 작업을 시작하지 않습니다.')
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timers = []
    if deadline:
        for delay, callback in [(max(0, deadline-time.time()-60), lambda: process.send_signal(signal.SIGINT)),
                                (max(0, deadline-time.time()), process.kill)]:
            timer = threading.Timer(delay, callback)
            timer.daemon = True
            timer.start()
            timers.append(timer)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('실행에 실패했거나 시간 예산이 끝났습니다. 위 결과를 확인하십시오.')
    finally:
        for timer in timers:
            timer.cancel()
UV = shutil.which('uv')
if UV is None:
    tool_env = Path('/content/vlade_uv_tool')
    tool_python = tool_env / 'bin/python'
    if not tool_python.is_file():
        run([sys.executable, '-m', 'venv', tool_env])
    run([tool_python, '-m', 'pip', 'install', 'uv'])
    UV = str(tool_env / 'bin/uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

# HF inputs are cached on the runtime SSD; Drive retains code and experiment records.
ASSET_CACHE = Path('/content/vlade_hf_assets')
ASSET_WORK_ROOT = Path('/content/vlade_asset_work')
RUNS_ROOT = Path('/content/vlade_runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
ENV.update(VLADE_ASSET_CACHE=str(ASSET_CACHE),
           VLADE_ASSET_REGISTRY=str(CODE/'configs/storage/hf_assets.lock.json'),
           HF_HOME='/content/vlade_hf_home', HF_TOKEN_PATH='/content/vlade_hf_token')
os.environ.update({key: ENV[key] for key in ('VLADE_ASSET_CACHE','VLADE_ASSET_REGISTRY')})
_HF_REGISTRY_READY = False

def hf_input(key, python=None, *, fetch=True):
    global _HF_REGISTRY_READY
    from vlade.storage.assets import asset_record, cache_path
    if fetch and not _HF_REGISTRY_READY:
        if python is None:
            raise ValueError('Select the prepared Python environment before fetching HF assets')
        registry = ASSET_CACHE / 'registry.json'
        run([python, 'scripts/data/hf_assets.py', 'refresh-registry', '--output', registry], env=ENV)
        ENV['VLADE_ASSET_REGISTRY'] = str(registry)
        os.environ['VLADE_ASSET_REGISTRY'] = str(registry)
        _HF_REGISTRY_READY = True
    record = asset_record(key, Path(ENV['VLADE_ASSET_REGISTRY']))
    target = cache_path(key, record, ASSET_CACHE)
    if fetch:
        if python is None:
            raise ValueError('Select the prepared CPU/model Python environment before fetching HF assets')
        run([python, 'scripts/data/hf_assets.py', 'fetch', key,
             '--registry', ENV['VLADE_ASSET_REGISTRY'], '--cache', ASSET_CACHE], env=ENV)
    return target

def hf_path(value, python=None, *, fetch=True):
    value = str(value)
    return hf_input(value[len('hfasset:'):].strip('/\\'), python, fetch=fetch) if value.startswith('hfasset:') else Path(value)

In [ ]:
# HF 설정 후 기존 학습 셀을 실행하면 checkpoint 저장 시 자동 전송합니다.
# 저장소는 HF 웹에서 Private로 먼저 만들고 해당 저장소 쓰기 토큰을 Secrets에 등록하십시오.
if 'ENV' not in globals():
    ENV = os.environ.copy()
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
ENV['HF_TOKEN'] = os.environ['HF_TOKEN']
import datetime, uuid
ENABLE_HUB_UPLOAD = globals().get('ENABLE_HUB_UPLOAD', True)
HF_REPO_ID = 'VLADE-konkuk/VLADE-checkpoints'  # 조직 저장소; 이전 런타임 값도 갱신합니다.
HF_RUN_GROUP = globals().get('HF_RUN_GROUP', datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])  # 재개 시 기존 group 지정
if ENABLE_HUB_UPLOAD:
    if not HF_RUN_GROUP:
        raise ValueError('HF_RUN_GROUP을 지정하십시오. pilot/main은 출력 폴더명으로 구분합니다.')
    os.environ['VLADE_HUB_REPO'] = HF_REPO_ID
    os.environ['VLADE_HUB_RUN_ID'] = HF_RUN_GROUP
    os.environ['HF_TOKEN_PATH'] = '/content/vlade_hf_token'
    # 모델 캐시는 기존 위치를 유지하고 토큰 파일은 런타임 로컬에만 둡니다.
    ENV.update(HF_TOKEN=os.environ['HF_TOKEN'], VLADE_HUB_REPO=HF_REPO_ID,
               VLADE_HUB_RUN_ID=HF_RUN_GROUP, HF_TOKEN_PATH='/content/vlade_hf_token')
    print('Private Hub 업로드 설정:', HF_REPO_ID, HF_RUN_GROUP)
else:
    for key in ('VLADE_HUB_REPO', 'VLADE_HUB_RUN_ID'):
        ENV.pop(key, None)
        os.environ.pop(key, None)
    print('Hub 업로드 꺼짐: HF_RUN_GROUP 설정 후 ENABLE_HUB_UPLOAD=True로 이 셀을 다시 실행하십시오.')

In [ ]:
RUN_INSTALL = False
RUN_DOWNLOAD = False
RUN_SMOKE = False
RUN_OVERFIT = False
RUN_MAIN_KD = False
kd_defaults = json.loads((CODE / 'configs/experiments/qwen_action_kd.json').read_text())
TEACHER_CACHE_RELATIVE = kd_defaults['teacher_cache']  # 필요하면 T1-full-hidden으로 변경
TASK_WEIGHT = kd_defaults['task_weight']
ACTION_KD_WEIGHT = kd_defaults['action_kd_weight']

In [ ]:
if RUN_INSTALL:
    STUDENT_PY = environment('vlade_qwen_kd_py313', '3.13', [
        'torch==2.12.0', 'torchvision==0.27.0', 'transformers==5.9.0',
        'accelerate', 'pillow', 'numpy', 'sentencepiece'])
    run([STUDENT_PY, '-c', "import torch; from transformers import Qwen3_5Model; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,torch.cuda.get_device_name(0))"])
    (ROOT / 'student_environment_freeze.txt').write_text(subprocess.check_output(
        [str(UV), 'pip', 'freeze', '--python', str(STUDENT_PY)], text=True))
else:
    STUDENT_PY = Path('/content/vlade_qwen_kd_py313/bin/python')
    print('환경 설치 대기: RUN_INSTALL=False')

## Qwen 초기 가중치 준비
이 셀을 실행하면 공식 고정 revision의 모델 가중치 약 1.75 GB와 processor 자료를 Drive에 받습니다. 기존 로컬 폴더의 processor 자료만으로는 전체 학습을 실행할 수 없습니다. 오늘 다운로드한 것으로 기록하지 않습니다.

In [ ]:
QWEN = hf_input('qwen_initial', fetch=False)
if RUN_DOWNLOAD:
    program = "import sys; from pathlib import Path; sys.path.insert(0,'src'); from vlade.data.colab_assets import download; download('Qwen/Qwen3.5-0.8B','2fc06364715b967f1860aea9cf38778875588b17',Path(sys.argv[1]))"
    hf_input('qwen_initial', STUDENT_PY)
else:
    print('기존 Qwen 가중치 사용:', QWEN, '/ 다운로드 미실행')

In [ ]:
PREPARED = hf_input('prepared_d_seed42', STUDENT_PY)
from vlade.data.episode_data import EpisodeData, write_json
from vlade.training.run_control import epoch_budget, overfit_report, require_reports, planned_checkpoint_space, pilot_context
MAIN_EPOCHS = 1
# RUN_MAIN_KD는 실행 스위치 셀에서 지정합니다.
OVERFIT_MAX_RATIO = .8  # 초기/마지막 loss 중앙값의 비율: 파일럿 감소 검사 기준
SAVES_PER_EPOCH = 4
BUDGET = epoch_budget(EpisodeData(PREPARED), MAIN_EPOCHS, batch_size=1, accumulation=8)
RUN_NAME = 'qwen-action-kd-epoch-seed42'
CONFIGS = ROOT / 'run-configs'
CONFIGS.mkdir(exist_ok=True)
student = json.loads((CODE / 'configs/models/student/qwen3_5_0_8b.json').read_text())['student']
base = dict(method='action_kd', data=str(PREPARED), initial_checkpoint=str(QWEN), student=student,
            seed=42, batch_size=1, accumulation_steps=8, max_steps=BUDGET['optimizer_steps'],
            save_steps=max(1,(BUDGET['optimizer_steps']+MAIN_EPOCHS*SAVES_PER_EPOCH-1)//(MAIN_EPOCHS*SAVES_PER_EPOCH)),
            learning_rate=2e-5, weight_decay=.01, validation_queries=32, eval_steps=100, episode_limit=0,
            budget=BUDGET,
            output=str(RUNS_ROOT / 'student_action_kd' / RUN_NAME))
base.update(teacher_cache=str(TEACHER_CACHE_RELATIVE),
            task_weight=TASK_WEIGHT, action_kd_weight=ACTION_KD_WEIGHT)
print('증류 설정:', base['teacher_cache'], 'task=', TASK_WEIGHT, 'action KD=', ACTION_KD_WEIGHT)
def student_config(suffix, **updates):
    settings = {**base, **updates}
    settings['output'] = str(RUNS_ROOT / 'student_action_kd' / (RUN_NAME + suffix))
    path = CONFIGS / (RUN_NAME + suffix + '.json')
    if path.exists() and json.loads(path.read_text()) != settings:
        raise ValueError('기존 실행 설정 불일치: RUN_NAME을 바꾸십시오.')
    path.write_text(json.dumps(settings, indent=2))
    return path, Path(settings['output'])
def train_student(config_path, output):
    settings = json.loads(config_path.read_text())
    run([STUDENT_PY, 'scripts/train/run_student_kd.py', '--config', config_path, '--preflight'])
    checkpoints = completed_checkpoints(output)
    latest = checkpoints[-1] if checkpoints else None
    if latest and int(latest.name.split('-')[1]) >= settings['max_steps']:
        print('이미 완료된 학습:', latest)
        return
    command = [STUDENT_PY, 'scripts/train/run_student_kd.py', '--config', config_path]
    if latest:
        command += ['--resume', latest]
    run(command)

In [ ]:
smoke_config, smoke_output = student_config('-one-step', max_steps=1, save_steps=1,
    accumulation_steps=1, validation_queries=1)
if RUN_SMOKE:
    train_student(smoke_config, smoke_output)
    run([STUDENT_PY, 'scripts/train/check_student_checkpoint.py', '--checkpoint', smoke_output / 'step-0000001',
         '--data', PREPARED, '--config', smoke_config, '--report', smoke_output / 'reload_report.json'])
else:
    print('1-step/재로딩 대기: RUN_SMOKE=False')

## 작은 episode overfit 검사
같은 train episode를 반복하며 처음/마지막 5개 loss 중앙값의 감소 비율을 자동 검사합니다. 이 파일럿 기준은 완전 수렴이나 rollout 성공률을 의미하지 않습니다. 실패 보고서가 있으면 본 학습을 거부합니다. 실제 가중치의 입력 모달리티 영향은 별도 확인 대상입니다.

In [ ]:
overfit_config, overfit_output = student_config('-overfit', max_steps=50, save_steps=50,
    accumulation_steps=1, episode_limit=1, validation_queries=4)
if RUN_OVERFIT:
    train_student(overfit_config, overfit_output)
    by_step = {}
    for log in sorted(overfit_output.glob('metrics-*.jsonl'), key=lambda path: path.stat().st_mtime):
        for line in log.read_text().splitlines():
            row = json.loads(line)
            by_step[row['step']] = row
    report = overfit_report(list(by_step.values()), max_ratio=OVERFIT_MAX_RATIO)
    overfit_steps = json.loads(overfit_config.read_text())['max_steps']
    report['context'] = pilot_context(overfit_config, overfit_output / f'step-{overfit_steps:07d}',
        PREPARED, 'student', policy={'max_ratio': OVERFIT_MAX_RATIO, 'window': 5})
    write_json(overfit_output / 'overfit_report.json', report)
    print('파일럿 loss 감소 검사:', report)
else:
    print('작은 episode 점검 대기: RUN_OVERFIT=False')

In [ ]:
MAIN_CONFIG, MAIN_OUTPUT = student_config('')
if RUN_MAIN_KD:
    overfit_steps = json.loads(overfit_config.read_text())['max_steps']
    require_reports([smoke_output / 'reload_report.json', overfit_output / 'overfit_report.json'],
        [pilot_context(smoke_config, smoke_output / 'step-0000001', PREPARED, 'student'),
         pilot_context(overfit_config, overfit_output / f'step-{overfit_steps:07d}', PREPARED, 'student',
            policy={'max_ratio': OVERFIT_MAX_RATIO, 'window': 5})])
    checkpoints = completed_checkpoints(MAIN_OUTPUT)
    print('저장 공간 계획:', planned_checkpoint_space(RUNS_ROOT, smoke_output / 'step-0000001',
        base['max_steps'], base['save_steps'], int(checkpoints[-1].name.split('-')[1]) if checkpoints else 0))
    train_student(MAIN_CONFIG, MAIN_OUTPUT)
else:
    print('본 학습 대기: budget·reload·overfit 결과 확인 후 RUN_MAIN_KD=True로 실행하십시오.')
reports = [json.loads(path.read_text()) for path in sorted(MAIN_OUTPUT.glob('run-*.json'), key=lambda path: path.stat().st_mtime_ns)]
print(json.dumps(reports[-1] if reports else {}, indent=2, ensure_ascii=False))
print('Validation L1은 rollout 성공률이 아닙니다. 환경 평가와 최종 기준 성능 기록은 별도로 진행하십시오.')